In [2]:
import json
from bitarray import bitarray
import time
from tqdm import tqdm
import re
from datetime import datetime
import pytz
import sys
import argparse
import os
import copy
import json
from parse_logs import parse_txn, parse_taskgraph, parse_debug
import pandas as pd
from datetime import datetime
import re
from tqdm import tqdm
import ast
import numpy as np



In [3]:
def datestring_to_timestamp(datestring):
    eastern = pytz.timezone('US/Eastern')
    date_obj = datetime.strptime(datestring, "%Y/%m/%d %H:%M:%S.%f")
    localized_date = eastern.localize(date_obj)
    unix_timestamp = localized_date.timestamp()
    return unix_timestamp

In [4]:
def parse_txn(txn):
    task_info = {}
    library_info = {}
    worker_info = {}
    manager_info = {}
    worker_coremap = {}
    task_try_count = {}         # task_id -> try_count
    file_info = {}

    total_lines = 0
    with open(txn, 'r') as file:
        for line in file:
            total_lines += 1

    with open(txn, 'r') as file:
        pbar = tqdm(total=total_lines, desc="parsing transactions")
        for line in file:
            pbar.update(1)

            if line.startswith("#"):
                continue

            timestamp, _, category, obj_id, status, *info = line.split(maxsplit=5)

            try:
                timestamp = float(timestamp) / 1000000
            except ValueError:
                continue

            info = info[0] if info else "{}"
            if category == 'TASK':
                task_id = int(obj_id)
                if status == 'READY':
                    if task_id not in task_try_count:
                        task_try_count[task_id] = 1
                    else:
                        task = task_info[(task_id, task_try_count[task_id])]
                        task['when_next_ready'] = timestamp
                        # reset the coremap for the new try
                        for i in task['core_id']:
                            worker_coremap[task['worker_committed']][i] = 0
                        task_try_count[task_id] += 1
                    task_category = info.split()[0]
                    try_id = task_try_count[task_id]
                    resources_requested = json.loads(info.split(' ', 3)[-1])
                    task = {
                        'task_id': task_id,
                        'try_id': try_id,
                        'worker_id': -1,
                        'core_id': [],

                        # Timestamps throughout the task lifecycle
                        'when_ready': timestamp,          # ready status on the manager
                        'time_commit_start': None,              # start commiting to worker
                        'time_commit_end': None,                # end commiting to worker
                        'when_running': None,             # running status on worker
                        'time_worker_start': None,              # start executing on worker
                        'time_worker_end': None,                # end executing on worker
                        'when_waiting_retrieval': None,   # waiting for retrieval status on worker
                        'when_retrieved': None,           # retrieved status on worker
                        'when_done': None,                # done status on worker
                        'when_next_ready': None,          # only for on-worker failed tasks

                        'worker_committed': None,

                        'size_input_mgr': None,
                        'size_output_mgr': None,
                        'cores_requested': resources_requested.get("cores", [0, ""])[0],
                        'gpus_requested': resources_requested.get("gpus", [0, ""])[0],
                        'memory_requested(MB)': resources_requested.get("memory", [0, ""])[0],
                        'disk_requested(MB)': resources_requested.get("disk", [0, ""])[0],
                        'retrieved_status': None,
                        'done_status': None,
                        'done_code': None,
                        'category': task_category,

                        'input_files': [],
                        'output_files': [],
                        'is_recovery_task': False,

                    }
                    if task['cores_requested'] == 0:
                        task['cores_requested'] = 1
                    task_info[(task_id, try_id)] = task
                if status == 'RUNNING':
                    # a running task can be a library which does not have a ready status
                    resources_allocated = json.loads(info.split(' ', 3)[-1])
                    if task_id in task_try_count:
                        try_id = task_try_count[task_id]
                        task = task_info[(task_id, try_id)]
                        worker_hash = info.split()[0]
                        task['when_running'] = timestamp
                        task['worker_committed'] = worker_hash
                        task['time_commit_start'] = float(resources_allocated["time_commit_start"][0])
                        task['time_commit_end'] = float(resources_allocated["time_commit_end"][0])
                        task['size_input_mgr'] = float(resources_allocated["size_input_mgr"][0])
                        coremap = worker_coremap[worker_hash]
                        cores_found = 0
                        for i in range(1, len(coremap)):
                            if coremap[i] == 0:
                                coremap[i] = 1
                                task['core_id'].append(i)
                                cores_found += 1
                                if cores_found == task['cores_requested']:
                                    break
                    else:
                        library = {
                            'task_id': task_id,
                            'when_running': timestamp,
                            'time_commit_start': resources_allocated["time_commit_start"][0],
                            'time_commit_end': resources_allocated["time_commit_end"][0],
                            'when_sent': None,
                            'when_started': None,
                            'when_retrieved': None,
                            'worker_committed': info.split(' ', 3)[0],
                            'worker_id': -1,
                            'size_input_mgr': resources_allocated["size_input_mgr"][0],
                            'cores_requested': resources_allocated.get("cores", [0, ""])[0],
                            'gpus_requested': resources_allocated.get("gpus", [0, ""])[0],
                            'memory_requested(MB)': resources_allocated.get("memory", [0, ""])[0],
                            'disk_requested(MB)': resources_allocated.get("disk", [0, ""])[0],
                        }
                        library_info[task_id] = library
                if status == 'WAITING_RETRIEVAL':
                    if task_id in task_try_count:
                        task = task_info[(task_id, task_try_count[task_id])]
                        task['when_waiting_retrieval'] = timestamp
                        worker_hash = task['worker_committed']
                        for core in task['core_id']:
                            worker_coremap[worker_hash][core] = 0
                if status == 'RETRIEVED':
                    try:
                        resources_retrieved = json.loads(info.split(' ', 5)[-1])
                    except json.JSONDecodeError:
                        resources_retrieved = {}
                    if task_id in task_try_count:
                        task = task_info[(task_id, task_try_count[task_id])]
                        task['when_retrieved'] = timestamp
                        task['retrieved_status'] = status
                        task['time_worker_start'] = resources_retrieved.get("time_worker_start", [None])[0]
                        task['time_worker_end'] = resources_retrieved.get("time_worker_end", [None])[0]
                        task['size_output_mgr'] = resources_retrieved.get("size_output_mgr", [None])[0]
                    else:
                        library = library_info[task_id]
                        library['when_retrieved'] = timestamp
                if status == 'DONE':
                    done_info = info.split() if info else []
                    if task_id in task_try_count:
                        task = task_info[(task_id, task_try_count[task_id])]
                        worker_hash = task['worker_committed']
                        task['when_done'] = timestamp
                        task['done_status'] = done_info[0] if len(done_info) > 0 else None
                        task['done_code'] = done_info[1] if len(done_info) > 1 else None
                        worker_info[worker_hash]['tasks_done'] += 1
            if category == 'WORKER':
                if not obj_id.startswith('worker'):
                    continue
                if status == 'CONNECTION':
                    if obj_id not in worker_info:
                        worker_info[obj_id] = {
                            'time_connected': [timestamp],
                            'time_disconnected': [],
                            'worker_id': -1,
                            'worker_machine_name': None,
                            'worker_ip': None,
                            'worker_port': None,
                            'tasks_done': 0,
                            'cores': None,
                            'memory(MB)': None,
                            'disk(MB)': None,
                            'disk_update': {},
                        }
                    else:
                        worker_info[obj_id]['time_connected'].append(timestamp)
                elif status == 'DISCONNECTION':
                    worker_info[obj_id]['time_disconnected'].append(timestamp)
                elif status == 'RESOURCES':
                    # only parse the first resources reported
                    if worker_info[obj_id]['cores'] is not None:
                        continue
                    resources = json.loads(info)
                    cores, memory, disk = resources.get("cores", [0, ""])[0], resources.get("memory", [0, ""])[0], resources.get("disk", [0, ""])[0]
                    worker_info[obj_id]['cores'] = cores
                    worker_info[obj_id]['memory(MB)'] = memory
                    worker_info[obj_id]['disk(MB)'] = disk
                    # for calculating task core_id
                    worker_coremap[obj_id] = bitarray(cores + 1)
                    worker_coremap[obj_id].setall(0)
                elif status == 'TRANSFER' or status == 'CACHE_UPDATE':
                    if status == 'TRANSFER':
                        # don't consider transfer as of now
                        transfer_type, filename, size_in_bytes, wall_time, start_time = info.split(' ', 4)
                    elif status == 'CACHE_UPDATE':
                        transfer_type = 'CACHE_UPDATE'
                        filename, size_in_bytes, wall_time, start_time = info.split(' ', 3)
                    
                    start_time = float(start_time) / 1e6
                    wall_time = float(wall_time) / 1e6

                    # start time should be after the manager start time
                    if start_time < manager_info['time_start']:
                        # consider xxx.04224 and xxx.0 as the same time
                        if abs(start_time - manager_info['time_start']) < 1:
                            start_time = manager_info['time_start']
                        else:
                            print(f"Warning: {transfer_type} start time {start_time} is before manager start time {manager_info['time_start']}")

                    # update disk usage
                    size_in_bytes = int(size_in_bytes)
                    if filename not in worker_info[obj_id]['disk_update']:
                        # this is the first time the file is cached
                        worker_info[obj_id]["disk_update"][filename] = {
                            'size(MB)': size_in_bytes / 2**20,
                            'when_stage_in': [start_time],
                            'when_stage_out': [],
                        }
                    else:
                        worker_info[obj_id]['disk_update'][filename]['when_stage_in'].append(start_time)

            if category == 'LIBRARY':
                if status == 'SENT':
                    for library in library_info.values():
                        if library['task_id'] == obj_id:
                            library['when_sent'] = timestamp
                if status == 'STARTED':
                    for library in library_info.values():
                        if library['task_id'] == obj_id:
                            library['when_started'] = timestamp
            if category == 'MANAGER':
                if status == 'START':
                    manager_info = {
                        'time_start': timestamp,
                        'time_end': None,
                        'lifetime(s)': None,
                        'time_start_human': None,
                        'time_end_human': None,
                        'tasks_submitted': 0,
                        'tasks_done': 0,
                        'tasks_failed_on_manager': 0,
                        'tasks_failed_on_worker': 0,
                        'max_task_try_count': 0,
                        'total_workers': 0,
                        'active_workers': 0,
                        'max_concurrent_workers': 0,
                    }
                if status == 'END':
                    manager_info['time_end'] = timestamp
                    manager_info['lifetime(s)'] = round(manager_info['time_end'] - manager_info['time_start'], 2)
        pbar.close()

    return task_info, task_try_count, library_info, worker_info, manager_info



In [5]:
def parse_debug(debug, worker_info, task_info, task_try_count, manager_info):

    total_lines = 0
    with open(debug, 'r') as file:
        for line in file:
            total_lines += 1

    putting_file = False
    worker_address_hash_map = {}

    with open(debug, 'r') as file:
        pbar = tqdm(total=total_lines, desc="parsing debug")
        for line in file:
            pbar.update(1)
            parts = line.strip().split(" ")

            if "worker-id" in parts:
                worker_id_id = parts.index("worker-id")
                worker_hash = parts[worker_id_id + 1]
                worker_machine_name = parts[worker_id_id - 3]
                worker_ip, worker_port = parts[worker_id_id - 2][1:-2].split(':')
                worker_address_hash_map[(worker_ip, worker_port)] = worker_hash
                if worker_hash in worker_info:
                    worker_info[worker_hash]['worker_machine_name'] = worker_machine_name
                    worker_info[worker_hash]['worker_ip'] = worker_ip
                    worker_info[worker_hash]['worker_port'] = worker_port

            if "put" in parts:
                putting_file = True
                continue
            if putting_file:
                putting_file = False
                file_id = parts.index("file")
                filename = parts[file_id + 1]
                size = int(parts[file_id + 2]) / 2**20
                start_time = float(parts[file_id + 4])
                if start_time < manager_info['time_start']:
                    if abs(start_time - manager_info['time_start']) < 1:
                        start_time = manager_info['time_start']
                    else:
                        print(f"Warning: put start time {start_time} is before manager start time {manager_info['time_start']}")
                worker_ip, worker_port = parts[file_id - 1][1:-2].split(':')
                worker_hash = worker_address_hash_map[(worker_ip, worker_port)]
                if filename not in worker_info[worker_hash]['disk_update']:
                    worker_info[worker_hash]['disk_update'][filename] = {
                        'size(MB)': size,
                        'when_stage_in': [start_time],
                        'when_stage_out': [],
                    }
                else:
                    worker_info[worker_hash]['disk_update'][filename]['when_stage_in'].append(start_time)
                if (size != worker_info[worker_hash]['disk_update'][filename]['size(MB)']):
                    print("Warning: size mismatch for file", filename, "size in debug: ", size, "size in txn: ", worker_info[worker_hash]['disk_update'][filename]['size(MB)'])
            
            if "cache-update" in parts:
                # already handled in parse_txn
                continue

            if "unlink" in parts:
                unlink_id = parts.index("unlink")
                filename = parts[unlink_id + 1]
                worker_ip, worker_port = parts[unlink_id - 1][1:-2].split(':')
                datestring = parts[0] + " " + parts[1]
                timestamp = datestring_to_timestamp(datestring)
                worker_hash = worker_address_hash_map[(worker_ip, worker_port)]
                try:
                    worker_info[worker_hash]['disk_update'][filename]['when_stage_out'].append(timestamp)
                    if len(worker_info[worker_hash]['disk_update'][filename]['when_stage_out']) > len(worker_info[worker_hash]['disk_update'][filename]['when_stage_in']):
                        print(f"Warning: file {filename} stage out more than stage in for worker {worker_hash}")
                except KeyError:
                    pass
                    # print("Warning: file", filename, f"not found in disk_update for worker {worker_ip}:{worker_port} {worker_hash}")    

            if "Submitted" in parts and "recovery" in parts and "task" in parts:
                task_id = int(parts[parts.index("task") + 1])
                try_count = task_try_count[task_id]
                for try_id in range(1, try_count + 1):
                    task_info[(task_id, try_id)]['is_recovery_task'] = True
                    task_info[(task_id, try_id)]['category'] = "recovery_task"
        pbar.close()
    return worker_info


In [6]:
def parse_taskgraph(taskgraph, task_info, task_try_count):
    total_lines = 0
    with open(taskgraph, 'r') as file:
        for line in file:
            total_lines += 1

    with open(taskgraph, 'r') as file:
        pbar = tqdm(total=total_lines, desc="parsing taskgraph")
        for line in file:
            pbar.update(1)
            if '->' not in line:
                continue
            left, right = line.split(' -> ')
            left = left.strip().strip('"')
            right = right.strip()[:-1].strip('"')

            # task produces an output file
            if left.startswith('task'):
                filename = right.split('-', 1)[1]
                task_id = int(left.split('-')[1])
                try_id = task_try_count[task_id]
                task_info[(task_id, try_id)]['output_files'].append(filename)
            # task consumes an input file
            elif right.startswith('task'):
                filename = left.split('-', 1)[1]
                task_id = int(right.split('-')[1])
                try_id = task_try_count[task_id]
                task_info[(task_id, try_id)]['input_files'].append(filename)
        pbar.close()

    return task_info

In [7]:
def expand_done_task(task):
    cores = task['core_id']
    if len(cores) == 0:
        task['core_id'] = -1
        return task
    elif len(cores) == 1:
        task['core_id'] = cores[0]
        return task
    else:
        print("Warning: task has multiple cores.")
        task_copies = []
        for core in cores:
            task_copy = task.copy()
            task_copy['core_id'] = core
            task_copies.append(task_copy)

        return task_copies

In [8]:
def remove_invalid_workers(worker_info, task_info, library_info):
    num_total_workers = len(worker_info)
    active_workers = set()
    for task in task_info.values():
        active_workers.add(task['worker_committed'])
    worker_info = {worker_hash: worker for worker_hash, worker in worker_info.items() if worker_hash in active_workers}
    num_active_workers = len(worker_info)
    # Sort workers by time connected
    worker_info = {k: v for k, v in sorted(worker_info.items(), key=lambda item: item[1]['time_connected'])}
    # Add worker_id to worker_info and update that in task_info and library_info
    worker_id = 1
    for worker in worker_info.values():
        worker['worker_id'] = worker_id
        worker_id += 1
    for task in task_info.values():
        if task['worker_committed']:
            task['worker_id'] = worker_info[task['worker_committed']]['worker_id']
    for library in library_info.values():
        if library['worker_committed']:
            library['worker_id'] = worker_info[library['worker_committed']]['worker_id']

    return worker_info, num_total_workers, num_active_workers

In [9]:
def generate_general_statistics(task_df, worker_summary_df, manager_info, num_total_workers, num_active_workers, dirname):
    #####################################################
    # General Statistics
    print("Generating general_statistics.csv...")
    general_statistics_task_df = task_df.groupby('category').agg({
        'task_id': 'nunique',
        'when_ready': lambda x: (x > 0).sum(),
        'when_running': lambda x: (x > 0).sum(),
        'when_waiting_retrieval': lambda x: (x > 0).sum(),
        'when_retrieved': lambda x: (x > 0).sum(),
        'when_done': lambda x: (x > 0).sum(),
        'worker_id': 'nunique',
    }).rename(columns={
        'task_id': 'submitted',
        'when_ready': 'ready',
        'when_running': 'running',
        'when_waiting_retrieval': 'waiting_retrieval',
        'when_retrieved': 'retrieved',
        'when_done': 'done',
        'worker_id': 'workers',
    }).reset_index()
    total_df = pd.DataFrame(columns=general_statistics_task_df.columns)
    total_df.loc[0, 'category'] = 'TOTAL'
    for col in ['submitted', 'ready', 'running', 'waiting_retrieval', 'retrieved', 'done']:
        total_df.loc[0, col] = general_statistics_task_df[col].sum()
    total_df.loc[0, 'workers'] = task_df['worker_id'].nunique()
    general_statistics_task_df = pd.concat([general_statistics_task_df, total_df], ignore_index=True)
    general_statistics_task_df = general_statistics_task_df.sort_values('submitted', ascending=False)
    general_statistics_task_df.to_csv(os.path.join(dirname, 'general_statistics_task.csv'), index=False)

    general_statistics_worker_df = pd.DataFrame(worker_summary_df)
    # convert time_connected and time_disconnected to datetime
    general_statistics_worker_df['time_connected'] = general_statistics_worker_df['time_connected'].apply(int)
    general_statistics_worker_df['time_disconnected'] = general_statistics_worker_df['time_disconnected'].apply(int)
    general_statistics_worker_df['time_connected'] = pd.to_datetime(general_statistics_worker_df['time_connected'], unit='s')
    general_statistics_worker_df['time_disconnected'] = pd.to_datetime(general_statistics_worker_df['time_disconnected'], unit='s')
    # round the values
    general_statistics_worker_df[['avg_task_runtime(s)', 'peak_disk_usage(MB)', 'peak_disk_usage(%)', 'lifetime(s)']] = general_statistics_worker_df[['avg_task_runtime(s)', 'peak_disk_usage(MB)', 'peak_disk_usage(%)', 'lifetime(s)']].round(2)
    general_statistics_worker_df.to_csv(os.path.join(dirname, 'general_statistics_worker.csv'), index=False)
    #####################################################

    #####################################################
    # Add info into manager_info
    print("Generating general_statistics_manager.csv...")
    manager_info['total_workers'] = num_total_workers
    manager_info['active_workers'] = num_active_workers

    events = pd.concat([
        pd.DataFrame({'time': worker_summary_df['time_connected'], 'type': 'connect'}),
        pd.DataFrame({'time': worker_summary_df['time_disconnected'], 'type': 'disconnect'})
    ])

    events = events.sort_values('time')
    current_workers = 0
    max_concurrent_workers = 0
    for _, event in events.iterrows():
        if event['type'] == 'connect':
            current_workers += 1
        else:
            current_workers -= 1
        max_concurrent_workers = max(max_concurrent_workers, current_workers)
    manager_info['max_concurrent_workers'] = max_concurrent_workers
    row_task_total = general_statistics_task_df[general_statistics_task_df['category'] == 'TOTAL']
    manager_info['tasks_submitted'] = row_task_total['submitted'].iloc[0]
    manager_info['time_start_human'] = pd.to_datetime(int(manager_info['time_start']), unit='s').strftime('%Y-%m-%d %H:%M:%S')
    manager_info['time_end_human'] = pd.to_datetime(int(manager_info['time_end']), unit='s').strftime('%Y-%m-%d %H:%M:%S')

    # the max try_id in task_df
    manager_info['max_task_try_count'] = task_df['try_id'].max()
    manager_info_df = pd.DataFrame([manager_info])
    manager_info_df.to_csv(os.path.join(dirname, 'general_statistics_manager.csv'), index=False)
    #####################################################

In [10]:
def generate_library_summary(library_info, dirname):
    library_df = pd.DataFrame.from_dict(library_info, orient='index')
    library_df.to_csv(os.path.join(dirname, 'library_summary.csv'), index=False)


In [11]:
def generate_worker_summary(worker_info, task_df, worker_disk_usage_df, dirname):
    import time
    time_start = time.time()
    print("Generating worker_summary.csv...")
    rows = []
    for worker_hash, info in worker_info.items():
        row = {
            'worker_id': info['worker_id'],
            'worker_hash': worker_hash,
            'worker_machine_name': info['worker_machine_name'],
            'worker_ip': info['worker_ip'],
            'worker_port': info['worker_port'],
            'time_connected': info['time_connected'],
            'time_disconnected': info['time_disconnected'],
            'lifetime(s)': 0,
            'cores': info['cores'],
            'memory(MB)': info['memory(MB)'],
            'disk(MB)': info['disk(MB)'],
            'tasks_done': 0,
            'avg_task_runtime(s)': 0,
            'peak_disk_usage(MB)': 0,
            'peak_disk_usage(%)': 0,
        }
        # calculate the number of tasks done by this worker
        tasks_committed = task_df[task_df['worker_committed'] == worker_hash]
        tasks_done = tasks_committed[tasks_committed['when_done'] > 0]
        row['tasks_done'] = len(tasks_done)
        # check if this worker has any disk updates
        if not worker_disk_usage_df.empty and worker_disk_usage_df['worker_hash'].isin([worker_hash]).any():
            row['peak_disk_usage(MB)'] = worker_disk_usage_df[worker_disk_usage_df['worker_hash'] == worker_hash]['disk_usage(MB)'].max()
            row['peak_disk_usage(%)'] = worker_disk_usage_df[worker_disk_usage_df['worker_hash'] == worker_hash]['disk_usage(%)'].max()
        # the worker may not have any tasks
        if row['tasks_done'] > 0:
            row['avg_task_runtime(s)'] = task_df[task_df['worker_committed'] == worker_hash]['time_worker_end'].mean() - task_df[task_df['worker_committed'] == worker_hash]['time_worker_start'].mean()
        if len(info['time_connected']) != len(info['time_disconnected']):
            raise ValueError("time_connected and time_disconnected have different lengths.")
        for i in range(len(info['time_connected'])):
            row_copy = copy.deepcopy(row)
            row_copy['time_connected'] = info['time_connected'][i]
            row_copy['time_disconnected'] = info['time_disconnected'][i]
            row_copy['lifetime(s)'] = info['time_disconnected'][i] - info['time_connected'][i]
            rows.append(row_copy)

    worker_summary_df = pd.DataFrame(rows)
    worker_summary_df = worker_summary_df.sort_values(by=['worker_id'], ascending=[True])
    worker_summary_df.to_csv(os.path.join(dirname, 'worker_summary.csv'), index=False)

    print(f"Time taken: {time.time() - time_start}")
    return worker_summary_df

In [12]:
def convert_to_and_save_task_df(task_info, manager_info, dirname):
    print("Generating task.csv...")

    task_df = pd.DataFrame.from_dict(task_info, orient='index')
    # ensure that the running time is not greater than the done time
    task_df['when_running'] = np.where(
        task_df['time_worker_start'].gt(0) & task_df['time_worker_start'].notna(),
        np.minimum(task_df['when_running'], task_df['time_worker_start']),
        task_df['when_running']
    )

    task_df.to_csv(os.path.join(dirname, 'task.csv'), index=False)

    is_done = task_df['when_done'].notnull()
    is_failed_manager = task_df['when_running'].isnull() & task_df['when_ready'].notnull()
    is_failed_worker = task_df['when_running'].notnull() & task_df['when_done'].isnull()

    manager_info['tasks_done'] = is_done.sum()
    manager_info['tasks_failed_on_manager'] = is_failed_manager.sum()
    manager_info['tasks_failed_on_worker'] = is_failed_worker.sum()

    task_df[is_done].apply(expand_done_task, axis=1).to_csv(os.path.join(dirname, 'task_done.csv'), index=False)
    task_df[is_failed_manager].apply(expand_done_task, axis=1).to_csv(os.path.join(dirname, 'task_failed_on_manager.csv'), index=False)
    task_df[is_failed_worker].apply(expand_done_task, axis=1).to_csv(os.path.join(dirname, 'task_failed_on_worker.csv'), index=False)

    return task_df

In [13]:
def generate_worker_disk_usage_df(worker_info, dirname):
    print("Generating worker_disk_usage.csv...")
    rows = []
    for worker_hash, worker in worker_info.items():
        for filename, disk_update in worker['disk_update'].items():
            # Initial checks for disk update logs
            len_in = len(disk_update['when_stage_in'])
            len_out = len(disk_update['when_stage_out'])
            if len_in < len_out:
                print(f"Warning: worker {worker_hash} has more stage-outs than stage-ins on file {filename}.")
            if len_in > len_out:
                # manually add a stage-out at the end of the log
                when_stage_in = disk_update['when_stage_in'][-1]
                worker_connected_id = next((i for i, connected in enumerate(worker['time_connected'])
                                            if connected < when_stage_in and worker['time_disconnected'][i] > when_stage_in), None)
                if worker_connected_id is not None:
                    disk_update['when_stage_out'].append(worker['time_disconnected'][worker_connected_id])

            # Preparing row data
            for time_point, size_modifier in zip(disk_update['when_stage_in'] + disk_update['when_stage_out'],
                                                 [disk_update['size(MB)']] * len_in + [-disk_update['size(MB)']] * len_out):
                rows.append({
                    'worker_hash': worker_hash,
                    'worker_id': worker['worker_id'],
                    'filename': filename,
                    'time': time_point,
                    'size(MB)': size_modifier
                })

    worker_disk_usage_df = pd.DataFrame(rows)

    if not worker_disk_usage_df.empty:
        worker_disk_usage_df = worker_disk_usage_df[worker_disk_usage_df['time'] > 0]
        worker_disk_usage_df.sort_values(by=['worker_id', 'time'], ascending=[True, True], inplace=True)
        worker_disk_usage_df['disk_usage(MB)'] = worker_disk_usage_df.groupby('worker_id')['size(MB)'].cumsum()
        worker_disk_usage_df['disk_usage(%)'] = worker_disk_usage_df['disk_usage(MB)'] / worker_disk_usage_df['worker_hash'].map(lambda x: worker_info[x]['disk(MB)'])
        worker_disk_usage_df.to_csv(os.path.join(dirname, 'worker_disk_usage.csv'), index=False)

    return worker_disk_usage_df

In [14]:
with os.scandir('logs') as entries:
    for entry in entries:
        if entry.is_dir():
            log_dir = os.path.join('logs', entry.name)
            break

dirname = os.path.join(log_dir, 'vine-logs')
txn = os.path.join(dirname, 'transactions')
debug = os.path.join(dirname, 'debug')
taskgraph = os.path.join(dirname, 'taskgraph')

task_info, task_try_count, library_info, worker_info, manager_info = parse_txn(txn)
task_info = parse_taskgraph(taskgraph, task_info, task_try_count)
worker_info = parse_debug(debug, worker_info, task_info, task_try_count, manager_info)

worker_info, num_total_workers, num_active_workers = remove_invalid_workers(worker_info, task_info, library_info)
task_df = convert_to_and_save_task_df(task_info, manager_info, dirname)

with open(os.path.join(dirname, 'worker_info.json'), 'w') as f:
    json.dump(worker_info, f, indent=4)

generate_library_summary(library_info, dirname)

worker_disk_usage_df = generate_worker_disk_usage_df(worker_info, dirname)

worker_summary_df = generate_worker_summary(worker_info, task_df, worker_disk_usage_df, dirname)

generate_general_statistics(task_df, worker_summary_df, manager_info, num_total_workers, num_active_workers, dirname)


parsing debug:   4%|▍         | 116355/3046722 [00:00<00:04, 587935.20it/s]

parsing debug: 100%|██████████| 3046722/3046722 [00:07<00:00, 397271.71it/s]


Generating task.csv...
Generating worker_disk_usage.csv...
Generating worker_summary.csv...
Time taken: 10.626442909240723
Generating general_statistics.csv...
Generating general_statistics_manager.csv...


In [15]:
def parse_txn(txn):
    task_info = {}
    library_info = {}
    worker_info = {}
    manager_info = {}
    worker_coremap = {}
    task_try_count = {}         # task_id -> try_count
    file_info = {}

    total_lines = 0
    with open(txn, 'r') as file:
        for line in file:
            total_lines += 1

    with open(txn, 'r') as file:
        pbar = tqdm(total=total_lines, desc="parsing transactions")
        for line in file:
            pbar.update(1)

            if line.startswith("#"):
                continue

            timestamp, _, category, obj_id, status, *info = line.split(maxsplit=5)

            try:
                timestamp = float(timestamp) / 1000000
            except ValueError:
                continue

            info = info[0] if info else "{}"
            if category == 'TASK':
                task_id = int(obj_id)
                if status == 'READY':
                    if task_id not in task_try_count:
                        task_try_count[task_id] = 1
                    else:
                        task = task_info[(task_id, task_try_count[task_id])]
                        task['when_next_ready'] = timestamp
                        # reset the coremap for the new try
                        for i in task['core_id']:
                            worker_coremap[task['worker_committed']][i] = 0
                        task_try_count[task_id] += 1
                    task_category = info.split()[0]
                    try_id = task_try_count[task_id]
                    resources_requested = json.loads(info.split(' ', 3)[-1])
                    task = {
                        'task_id': task_id,
                        'try_id': try_id,
                        'worker_id': -1,
                        'core_id': [],

                        # Timestamps throughout the task lifecycle
                        'when_ready': timestamp,          # ready status on the manager
                        'time_commit_start': None,              # start commiting to worker
                        'time_commit_end': None,                # end commiting to worker
                        'when_running': None,             # running status on worker
                        'time_worker_start': None,              # start executing on worker
                        'time_worker_end': None,                # end executing on worker
                        'when_waiting_retrieval': None,   # waiting for retrieval status on worker
                        'when_retrieved': None,           # retrieved status on worker
                        'when_done': None,                # done status on worker
                        'when_next_ready': None,          # only for on-worker failed tasks

                        'worker_committed': None,

                        'size_input_mgr': None,
                        'size_output_mgr': None,
                        'cores_requested': resources_requested.get("cores", [0, ""])[0],
                        'gpus_requested': resources_requested.get("gpus", [0, ""])[0],
                        'memory_requested(MB)': resources_requested.get("memory", [0, ""])[0],
                        'disk_requested(MB)': resources_requested.get("disk", [0, ""])[0],
                        'retrieved_status': None,
                        'done_status': None,
                        'done_code': None,
                        'category': task_category,

                        'input_files': [],
                        'output_files': [],
                        'is_recovery_task': False,

                    }
                    if task['cores_requested'] == 0:
                        task['cores_requested'] = 1
                    task_info[(task_id, try_id)] = task
                if status == 'RUNNING':
                    # a running task can be a library which does not have a ready status
                    resources_allocated = json.loads(info.split(' ', 3)[-1])
                    if task_id in task_try_count:
                        try_id = task_try_count[task_id]
                        task = task_info[(task_id, try_id)]
                        worker_hash = info.split()[0]
                        task['when_running'] = timestamp
                        task['worker_committed'] = worker_hash
                        task['time_commit_start'] = float(resources_allocated["time_commit_start"][0])
                        task['time_commit_end'] = float(resources_allocated["time_commit_end"][0])
                        task['size_input_mgr'] = float(resources_allocated["size_input_mgr"][0])
                        coremap = worker_coremap[worker_hash]
                        cores_found = 0
                        for i in range(1, len(coremap)):
                            if coremap[i] == 0:
                                coremap[i] = 1
                                task['core_id'].append(i)
                                cores_found += 1
                                if cores_found == task['cores_requested']:
                                    break
                    else:
                        library = {
                            'task_id': task_id,
                            'when_running': timestamp,
                            'time_commit_start': resources_allocated["time_commit_start"][0],
                            'time_commit_end': resources_allocated["time_commit_end"][0],
                            'when_sent': None,
                            'when_started': None,
                            'when_retrieved': None,
                            'worker_committed': info.split(' ', 3)[0],
                            'worker_id': -1,
                            'size_input_mgr': resources_allocated["size_input_mgr"][0],
                            'cores_requested': resources_allocated.get("cores", [0, ""])[0],
                            'gpus_requested': resources_allocated.get("gpus", [0, ""])[0],
                            'memory_requested(MB)': resources_allocated.get("memory", [0, ""])[0],
                            'disk_requested(MB)': resources_allocated.get("disk", [0, ""])[0],
                        }
                        library_info[task_id] = library
                if status == 'WAITING_RETRIEVAL':
                    if task_id in task_try_count:
                        task = task_info[(task_id, task_try_count[task_id])]
                        task['when_waiting_retrieval'] = timestamp
                        worker_hash = task['worker_committed']
                        for core in task['core_id']:
                            worker_coremap[worker_hash][core] = 0
                if status == 'RETRIEVED':
                    try:
                        resources_retrieved = json.loads(info.split(' ', 5)[-1])
                    except json.JSONDecodeError:
                        resources_retrieved = {}
                    if task_id in task_try_count:
                        task = task_info[(task_id, task_try_count[task_id])]
                        task['when_retrieved'] = timestamp
                        task['retrieved_status'] = status
                        task['time_worker_start'] = resources_retrieved.get("time_worker_start", [None])[0]
                        task['time_worker_end'] = resources_retrieved.get("time_worker_end", [None])[0]
                        task['size_output_mgr'] = resources_retrieved.get("size_output_mgr", [None])[0]
                    else:
                        library = library_info[task_id]
                        library['when_retrieved'] = timestamp
                if status == 'DONE':
                    done_info = info.split() if info else []
                    if task_id in task_try_count:
                        task = task_info[(task_id, task_try_count[task_id])]
                        worker_hash = task['worker_committed']
                        task['when_done'] = timestamp
                        task['done_status'] = done_info[0] if len(done_info) > 0 else None
                        task['done_code'] = done_info[1] if len(done_info) > 1 else None
                        worker_info[worker_hash]['tasks_done'] += 1
            if category == 'WORKER':
                if not obj_id.startswith('worker'):
                    continue
                if status == 'CONNECTION':
                    if obj_id not in worker_info:
                        worker_info[obj_id] = {
                            'time_connected': [timestamp],
                            'time_disconnected': [],
                            'worker_id': -1,
                            'worker_machine_name': None,
                            'worker_ip': None,
                            'worker_port': None,
                            'tasks_done': 0,
                            'cores': None,
                            'memory(MB)': None,
                            'disk(MB)': None,
                            'disk_update': {},
                        }
                    else:
                        worker_info[obj_id]['time_connected'].append(timestamp)
                elif status == 'DISCONNECTION':
                    worker_info[obj_id]['time_disconnected'].append(timestamp)
                elif status == 'RESOURCES':
                    # only parse the first resources reported
                    if worker_info[obj_id]['cores'] is not None:
                        continue
                    resources = json.loads(info)
                    cores, memory, disk = resources.get("cores", [0, ""])[0], resources.get("memory", [0, ""])[0], resources.get("disk", [0, ""])[0]
                    worker_info[obj_id]['cores'] = cores
                    worker_info[obj_id]['memory(MB)'] = memory
                    worker_info[obj_id]['disk(MB)'] = disk
                    # for calculating task core_id
                    worker_coremap[obj_id] = bitarray(cores + 1)
                    worker_coremap[obj_id].setall(0)
                elif status == 'TRANSFER' or status == 'CACHE_UPDATE':
                    if status == 'TRANSFER':
                        # don't consider transfer as of now
                        transfer_type, filename, size_in_bytes, wall_time, start_time = info.split(' ', 4)
                    elif status == 'CACHE_UPDATE':
                        transfer_type = 'CACHE_UPDATE'
                        filename, size_in_bytes, wall_time, start_time = info.split(' ', 3)

                    start_time = float(start_time) / 1e6
                    wall_time = float(wall_time) / 1e6

                    # start time should be after the manager start time
                    if start_time < manager_info['time_start']:
                        # consider xxx.04224 and xxx.0 as the same time
                        if abs(start_time - manager_info['time_start']) < 1:
                            start_time = manager_info['time_start']
                        else:
                            print(f"Warning: {transfer_type} start time {start_time} is before manager start time {manager_info['time_start']}")

                    # update disk usage
                    size_in_bytes = int(size_in_bytes)
                    if filename not in worker_info[obj_id]['disk_update']:
                        # this is the first time the file is cached
                        worker_info[obj_id]["disk_update"][filename] = {
                            'size(MB)': size_in_bytes / 2**20,
                            'when_stage_in': [start_time],
                            'when_stage_out': [],
                        }
                    else:
                        worker_info[obj_id]['disk_update'][filename]['when_stage_in'].append(start_time)

            if category == 'LIBRARY':
                if status == 'SENT':
                    for library in library_info.values():
                        if library['task_id'] == obj_id:
                            library['when_sent'] = timestamp
                if status == 'STARTED':
                    for library in library_info.values():
                        if library['task_id'] == obj_id:
                            library['when_started'] = timestamp
            if category == 'MANAGER':
                if status == 'START':
                    manager_info = {
                        'time_start': timestamp,
                        'time_end': None,
                        'lifetime(s)': None,
                        'time_start_human': None,
                        'time_end_human': None,
                        'tasks_submitted': 0,
                        'tasks_done': 0,
                        'tasks_failed_on_manager': 0,
                        'tasks_failed_on_worker': 0,
                        'max_task_try_count': 0,
                        'total_workers': 0,
                        'active_workers': 0,
                        'max_concurrent_workers': 0,
                    }
                if status == 'END':
                    manager_info['time_end'] = timestamp
                    manager_info['lifetime(s)'] = round(manager_info['time_end'] - manager_info['time_start'], 2)
        pbar.close()

    return task_info, task_try_count, library_info, worker_info, manager_info



task_df.length

In [58]:
def generate_worker_disk_usage_df(worker_info, file_info, dirname):
    print("Generating worker_disk_usage.csv...")
    rows = []
    for worker_hash, worker in worker_info.items():
        worker_id = worker['worker_id']
        for filename, disk_update in worker['disk_update'].items():
            # Initial checks for disk update logs
            len_in = len(disk_update['when_stage_in'])
            len_out = len(disk_update['when_stage_out'])
            if len_in < len_out:
                print(f"Warning: worker {worker_hash} has more stage-outs than stage-ins on file {filename}.")
            if len_in > len_out:
                # manually add a stage-out at the end of the log
                when_stage_in = disk_update['when_stage_in'][-1]
                worker_connected_id = next((i for i, connected in enumerate(worker['time_connected'])
                                            if connected < when_stage_in and worker['time_disconnected'][i] > when_stage_in), None)
                if worker_connected_id is not None:
                    disk_update['when_stage_out'].append(worker['time_disconnected'][worker_connected_id])
            if len_in != len_out:
                print(f"Warning: worker {worker_hash} has different number of stage-ins and stage-outs on file {filename}.")
            if filename in file_info:
                if 'size(MB)' not in file_info[filename]:
                    file_info[filename]['size(MB)'] = disk_update['size(MB)']
                    file_info[filename]['worker_holding'] = []
                else:
                    if file_info[filename]['size(MB)'] != disk_update['size(MB)']:
                        print(f"Warning: size mismatch for file {filename} size in disk_update: {disk_update['size(MB)']} size in file_info: {file_info[filename]['size(MB)']}")
                for i in range(len_in):
                    file_info[filename]['worker_holding'].append((worker_id, round(disk_update['when_stage_in'][i], 2), round(disk_update['when_stage_out'][i], 2)))

            # Preparing row data
            for time, disk_increment in zip(disk_update['when_stage_in'] + disk_update['when_stage_out'],
                                                 [disk_update['size(MB)']] * len_in + [-disk_update['size(MB)']] * len_out):
                rows.append({
                    'worker_hash': worker_hash,
                    'worker_id':worker_id,
                    'filename': filename,
                    'time': time,
                    'size(MB)': disk_increment
                })

    worker_disk_usage_df = pd.DataFrame(rows)

    if not worker_disk_usage_df.empty:
        worker_disk_usage_df = worker_disk_usage_df[worker_disk_usage_df['time'] > 0]
        worker_disk_usage_df.sort_values(by=['worker_id', 'time'], ascending=[True, True], inplace=True)
        worker_disk_usage_df['disk_usage(MB)'] = worker_disk_usage_df.groupby('worker_id')['size(MB)'].cumsum()
        worker_disk_usage_df['disk_usage(%)'] = worker_disk_usage_df['disk_usage(MB)'] / worker_disk_usage_df['worker_hash'].map(lambda x: worker_info[x]['disk(MB)'])
        worker_disk_usage_df.to_csv(os.path.join(dirname, 'worker_disk_usage.csv'), index=False)

    return worker_disk_usage_df, file_info

file_info = {}

for index, row in task_df.iterrows():
    # exclude recovery tasks 
    if row['category'] == 'recovery_task':
        continue
    task_id = row['task_id']
    input_files = row['input_files']
    output_files = row['output_files']

    for file in input_files:
        # exclude metadata files (only exist as input of some tasks)
        if file.startswith('file-meta'):
            continue
        if file not in file_info:
            file_info[file] = {'producers': [], 'consumers': []}
        file_info[file]['consumers'].append(task_id)

    for file in output_files:
        if file not in file_info:
            file_info[file] = {'producers': [], 'consumers': []}
        file_info[file]['producers'].append(task_id)


worker_summary_df, file_info  = generate_worker_disk_usage_df(worker_info, file_info, dirname)

data = []
for filename, info in file_info.items():
    data.append({
        'filename': filename,
        'size(MB)': round(info['size(MB)'], 6),
        'num_worker_holding': len(info['worker_holding']),
        'producers': info['producers'],
        'consumers': info['consumers']
    })
file_info_df = pd.DataFrame(data)
file_info_df.to_csv(os.path.join(dirname, 'general_statistics_file.csv'), index=False)


Generating worker_disk_usage.csv...
